# 01 — Detected-only data preparation and documented exclusions (V3)
**Wildlife confidence calibration | MobileNetV3-Large versus EfficientNet-B0 | five known animal categories**

This notebook prepares the third version of the wildlife confidence experiment. The research comparison remains the change in probability quality after temperature scaling for two compact pretrained classifiers. V3 addresses a preprocessing limitation identified in the preceding experiment: selected images without a qualifying MegaDetector animal box had been retained as full-frame fallbacks, although their species labels did not establish that the selected frame visibly contained that animal.

V3 retains **only `animal_crop` events** from the completed V2 sample. The rule applies consistently to training, validation, calibration, policy selection and final testing. Exclusion uses the existing detector decision, not classifier predictions, confidence, correctness or observed test loss. All retained photographs, event IDs, species labels, camera assignments and within-manifest order remain unchanged. Excluded observations are not replaced or topped up.

The five categories remain **coyote, deer, fox, rabbit and skunk**. Their decision predates the original camera allocation; they are not reselected after inspecting model performance. The original one-frame-per-unambiguous-trigger-event representation and camera-disjoint partitions are inherited from V2. V3 does not rerun the camera allocation optimizer or select another frame from an excluded event.

**Study population:** one selected photograph from a five-category event with a qualifying precomputed animal detection. A missing detection does not prove that the image is empty, and a predicted box does not prove that the crop contains the assigned species. Accordingly, this is a detected-only classifier study. It does not estimate detector recall, empty-scene detection, unknown-species recognition or automation coverage over an unfiltered camera stream.

Run this notebook before Notebooks 02–05 V3. It reads the completed local V2 preparation, writes a separate V3 dataset definition and exclusion audit, and performs no image download, model training or prediction. V3 was designed after examining V1/V2 results and retains previously examined camera assignments; its evaluation is a revised experiment rather than independent confirmation on fresh cameras.


## 1. Define the experiment, categories and reproducibility helpers

We give the revised study its own experiment identifier, `five_categories_detected_only_v3`, and retain the V2 class-index mapping. This ensures that classifier outputs refer to the same five categories throughout training, calibration and evaluation.

The helper functions calculate SHA-256 file hashes, serialize JSON consistently, write files through temporary paths, and freeze scientific inputs. A frozen file can be reused only when its contents agree exactly. A discrepancy stops execution rather than overwriting a completed study. Relative paths are checked to prevent input/output paths from leaving their intended project root. CSV identity fields are read as strings so that camera and event identifiers are preserved.


In [1]:
from pathlib import Path
import hashlib, json, shutil
import numpy as np
import pandas as pd

EXPERIMENT_ID = "five_categories_detected_only_v3"
PARENT_EXPERIMENT_ID = "five_categories_camera_balanced_v2"
CLASSES = ['coyote', 'deer', 'fox', 'rabbit', 'skunk']
SPLITS = ['train', 'validation', 'calibration', 'policy', 'test']

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def jtext(value):
    return json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + '\n'

def write(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(content, encoding='utf-8')
    temporary.replace(path)

def freeze(path, content):
    path = Path(path)
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise ValueError(f'Frozen V3 input differs: {path}. Preserve this run; use a new experiment for amendments.')
    if not path.exists():
        write(path, content)

def safe(base, relative):
    base = Path(base).resolve()
    relative = Path(relative)
    if relative.is_absolute() or '..' in relative.parts:
        raise ValueError(f'Unsafe relative path: {relative}')
    path = (base / relative).resolve()
    if not path.is_relative_to(base):
        raise ValueError(f'Path leaves project: {path}')
    return path

def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


## 2. Locate the read-only V2 source and separate V3 destination

`SOURCE_PROJECT_ROOT` refers to the completed **local V2 data root**, containing `data/prepared` and the processed image cache under `data/images`. In the repository layout this can be the inner `wildlife_confidence_project_v2` directory. Automatic discovery checks the current and neighbouring project locations; set the variable explicitly if there are no matches or several candidates.

`PROJECT_ROOT` specifies the new V3 destination. By default, running from the extracted `wildlife_confidence_project_v3` folder uses that folder; otherwise a V3 subdirectory is created. V2 and V3 must be separate, non-nested roots. The source is read only, and V3 preparation, models and reports have separate paths. The original V2 checkpoints, logits, temperatures, thresholds and results are not imported into the new experiment.


In [2]:
# Set this only if automatic discovery does not find your EXISTING V2 data root.
# It must contain data/prepared/five_categories_camera_balanced_v2/.
# In the GitHub layout this is the INNER wildlife_confidence_project_v2 folder.
SOURCE_PROJECT_ROOT = None
PROJECT_ROOT = None  # V3 output root; default is this notebook folder.

if PROJECT_ROOT is None:
    ROOT = (Path.cwd() if Path.cwd().name == 'wildlife_confidence_project_v3'
            else Path.cwd() / 'wildlife_confidence_project_v3').resolve()
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()

if SOURCE_PROJECT_ROOT is None:
    bases = [Path.cwd(), Path.cwd().parent, ROOT.parent]
    possible = []
    for base in bases:
        possible.extend([base, base / 'wildlife_confidence_project_v2',
                         base / 'wildlife_confidence_project_v2' / 'wildlife_confidence_project_v2'])
    found = list(dict.fromkeys(p.resolve() for p in possible if
        (p / 'data/prepared' / PARENT_EXPERIMENT_ID / 'training_ready_manifest.csv').is_file()))
    if len(found) != 1:
        raise FileNotFoundError('Set SOURCE_PROJECT_ROOT to your local V2 data root (including its image cache). '
                                f'Found {len(found)} candidates: {found}')
    SOURCE_ROOT = found[0]
else:
    SOURCE_ROOT = Path(SOURCE_PROJECT_ROOT).expanduser().resolve()
if ROOT == SOURCE_ROOT or ROOT.is_relative_to(SOURCE_ROOT) or SOURCE_ROOT.is_relative_to(ROOT):
    raise ValueError('V3 and V2 must be separate, non-nested project directories.')
PREPARED = ROOT / 'data/prepared'
EXP = PREPARED / EXPERIMENT_ID
REPORT = ROOT / 'reports' / EXPERIMENT_ID
PARENT_PREPARED = SOURCE_ROOT / 'data/prepared'
PARENT_EXP = PARENT_PREPARED / PARENT_EXPERIMENT_ID
for p in [EXP, REPORT]:
    p.mkdir(parents=True, exist_ok=True)
print('Read-only V2 source:', SOURCE_ROOT)
print('Separate V3 destination:', ROOT)


Read-only V2 source: /home/fia/Schreibtisch/wildlife_confidence_project_v2/wildlife_confidence_project_v2
Separate V3 destination: /home/fia/Schreibtisch/wildlife_confidence_project_v3


## 3. Verify inherited event, camera and detector provenance

Before changing the sample, verify V2's frozen input hashes, selected-manifest/ready-manifest agreement, one-image-per-event identities, detector-plan coverage and camera allocation. Selected labels, file names and partitions must agree with the original event manifest, and a camera may occur in only one partition.

Only the documented `animal_crop` and `full_frame_fallback` modes are accepted as source decisions. A detector-record error is a technical failure that must be investigated; it is not silently converted into an excluded no-detection observation. Source and cache image hashes must also be available for subsequent image verification.

The original preparation and camera records retain their V2 protocol identifiers because their bytes and assignments are inherited unchanged. Separate V3 selection and preprocessing protocols record the new study population. This preserves an honest provenance chain rather than relabelling an earlier split as a newly generated one. No classifier output or saved performance result is read in this step.


In [3]:
parent_protocol = json.loads((PARENT_EXP / 'protocol.json').read_text())
parent_selection = json.loads((PARENT_EXP / 'selection_protocol.json').read_text())
if (parent_protocol.get('pipeline_kind') != 'megadetector_assisted_single_v2'
        or parent_protocol.get('experiment_id') != PARENT_EXPERIMENT_ID
        or parent_protocol.get('classes') != CLASSES
        or parent_selection.get('experiment_id') != PARENT_EXPERIMENT_ID):
    raise ValueError('Expected the completed five-category V2 preparation.')
for relative, expected in parent_protocol['input_sha256'].items():
    if sha(safe(PARENT_PREPARED, relative)) != expected:
        raise ValueError(f'V2 provenance mismatch: {relative}')
parent_base = read_csv(PARENT_EXP / 'subset_manifest.csv')
parent_ready = read_csv(PARENT_EXP / 'training_ready_manifest.csv')
parent_plan = read_csv(PARENT_EXP / 'crop_plan.csv')
parent_source = read_csv(PARENT_PREPARED / 'image_manifest.csv')
camera = json.loads((PARENT_PREPARED / 'camera_split.json').read_text())
prep = json.loads((PARENT_PREPARED / 'preparation_run.json').read_text())
if (camera.get('protocol') != 'camera_stratified_event_v2'
        or camera.get('experiment_id') != PARENT_EXPERIMENT_ID
        or prep.get('stage') != '01_event_first_camera_split_v2'
        or prep.get('experiment_id') != PARENT_EXPERIMENT_ID
        or camera['selected_classes'] != CLASSES):
    raise ValueError('Unexpected V2 camera allocation or category decision.')
for name, expected in [('camera_split.json', prep['split_sha256']),
                       ('image_manifest.csv', prep['manifest_sha256']),
                       ('event_manifest_all_categories.csv', camera['event_manifest_sha256'])]:
    if sha(PARENT_PREPARED / name) != expected:
        raise ValueError(f'V2 source preparation changed: {name}')
if not parent_ready[parent_base.columns].equals(parent_base):
    raise ValueError('V2 ready manifest and selected manifest disagree.')
for frame in [parent_base, parent_ready, parent_plan]:
    if not frame.image_id.is_unique:
        raise ValueError('Duplicate V2 image IDs.')
if set(parent_plan.image_id) != set(parent_base.image_id):
    raise ValueError('V2 crop plan does not cover the exact selected events.')
if not parent_base.seq_id.is_unique or parent_base.groupby('location').split.nunique().gt(1).any():
    raise ValueError('V2 event duplication or camera leakage.')
if not parent_base.location.map(camera['assignments']).eq(parent_base.split).all():
    raise ValueError('V2 selected events disagree with inherited camera assignments.')
identities = ['file_name', 'location', 'seq_id', 'category_id', 'class_name', 'split']
if not (parent_base.set_index('image_id')[identities].equals(
        parent_source.set_index('image_id').loc[parent_base.image_id, identities])):
    raise ValueError('V2 selected event identity changed.')
if not parent_plan.set_index('image_id').loc[parent_base.image_id, 'file_name'].to_numpy().tolist() == parent_base.file_name.tolist():
    raise ValueError('V2 detector filename mismatch.')
if not set(parent_plan['mode']).issubset({'animal_crop', 'full_frame_fallback'}):
    raise ValueError('Detector errors or unknown crop modes: repair upstream; do not silently exclude technical errors.')
if parent_ready[['source_sha256','cache_sha256']].apply(lambda c: c.str.fullmatch(r'[a-f0-9]{64}')).ne(True).any().any():
    raise ValueError('Missing or malformed V2 image hashes.')
print('V2 preparation provenance verified; no model results were read.')


V2 preparation provenance verified; no model results were read.


## 4. Exclude full-frame fallbacks in all five partitions and audit support

Retain exactly those V2 selected events whose existing crop mode is `animal_crop`. Every other legitimate no-detection fallback is recorded in an exclusion table, with its image/event ID, category, camera, partition and reason. The code does not describe these frames as manually confirmed empty.

The detector rule remains the V2 rule: choose the highest-confidence animal box at confidence **at least 0.20**, breaking ties by coordinates; expand the chosen box by **5% per side**, clip it to the source image and aspect-resize/pad the crop to **224 × 224 RGB**. No species-label matching is used to choose the box. Multiple-box cases keep the same chosen box. The retained box coordinates, confidence, dimensions and detector status are checked for valid geometry.

Partition sizes shrink without resampling. For the supplied repository snapshot, the expected flow is:

| Partition | V2 events | Excluded fallbacks | V3 crops |
|---|---:|---:|---:|
| Training | 4,570 | 193 | 4,377 |
| Validation | 1,500 | 170 | 1,330 |
| Calibration | 1,500 | 148 | 1,352 |
| Policy | 1,500 | 111 | 1,389 |
| Test | 3,000 | 125 | 2,875 |

These are preparation counts, not new model results. The notebook prints counts from the actual source, checks that every class remains represented in every partition, and retains the development requirement of at least **40 observations per category** in validation, calibration and policy. Insufficient support stops preparation instead of silently changing the design.

Camera contributions are recomputed after exclusion. V2's development cap applied before filtering; removing events changes the denominator, so a retained camera share can exceed 20%. We report these actual shares and preserve the inherited absolute count limits. Test remains the retained subset of V2's event sample, without a new camera cap. Test counts are bookkeeping and do not select models or policies.


In [4]:
ordered_plan = parent_plan.set_index('image_id').loc[parent_base.image_id].reset_index()
keep = ordered_plan['mode'].eq('animal_crop').to_numpy()
subset = parent_base.loc[keep].copy().reset_index(drop=True)
plan = ordered_plan.loc[keep].copy().reset_index(drop=True)
excluded = parent_base.loc[~keep, ['image_id','file_name','seq_id','class_name','location','split']].copy()
excluded['reason'] = 'no_qualifying_animal_detection_full_frame_fallback'
excluded['confirmed_empty'] = 'not_manually_verified'

# Exactly the V2 crop-only subset, preserving labels, cameras, event IDs and row order.
# No new samples, replacement frames, camera reassignment or test-driven balancing.
subset['cache_relative_path'] = subset.image_id.map(
    lambda image_id: f'data/images/{EXPERIMENT_ID}/{image_id}.png')
if not subset.label.astype(int).eq(subset.class_name.map({c:i for i,c in enumerate(CLASSES)})).all():
    raise ValueError('Incorrect class indices.')
boxes = plan[['x','y','width','height','confidence']].astype(float)
if (not np.isfinite(boxes.to_numpy()).all() or (boxes[['width','height']] <= 0).any().any()
        or (boxes[['x','y']] < 0).any().any()
        or (boxes.x + boxes.width > 1 + 1e-12).any()
        or (boxes.y + boxes.height > 1 + 1e-12).any()
        or not boxes.confidence.between(0.20, 1).all()
        or not plan.animal_box_count.astype(int).ge(1).all() or plan.error.ne('').any()):
    raise ValueError('A retained crop lacks a valid qualifying animal box.')
counts = pd.crosstab(subset.class_name, subset.split).reindex(index=CLASSES, columns=SPLITS, fill_value=0)
if counts.eq(0).any().any() or counts[['validation','calibration','policy']].lt(40).any().any():
    raise ValueError('Crop-only class support fails the retained 40-event development minimum. Do not silently resample.')
sizes = subset.groupby('split').size().reindex(SPLITS)
flow = pd.DataFrame({'v2_events': parent_base.groupby('split').size(),
                     'excluded_fallbacks': excluded.groupby('split').size(),
                     'v3_events': sizes}).fillna(0).astype(int).reindex(SPLITS)
by_class = (parent_base.groupby(['split','class_name']).size().rename('v2_events').to_frame()
            .join(excluded.groupby(['split','class_name']).size().rename('excluded_fallbacks'))
            .join(subset.groupby(['split','class_name']).size().rename('v3_events')).fillna(0).astype(int).reset_index())
camera_counts = subset.groupby(['split','location']).size().rename('events').reset_index()
camera_counts['split_events'] = camera_counts.split.map(sizes)
camera_counts['share'] = camera_counts.events / camera_counts.split_events
print(flow.to_string())
print('Retained class counts:\n', counts.to_string())
print('Actual largest camera share after exclusions:', camera_counts.groupby('split').share.max().round(4).to_dict())
print('Counts shrink after filtering; the old 20% share cap need not hold against the smaller denominator.')


             v2_events  excluded_fallbacks  v3_events
split                                                
train             4570                 193       4377
validation        1500                 170       1330
calibration       1500                 148       1352
policy            1500                 111       1389
test              3000                 125       2875
Retained class counts:
 split       train  validation  calibration  policy  test
class_name                                              
coyote       1177         498          563     558  1292
deer         1166         386          396     382   715
fox           411          60           55      51    93
rabbit       1137         334          290     342   687
skunk         486          52           48      56    88
Actual largest camera share after exclusions: {'calibration': 0.199, 'policy': 0.2131, 'test': 0.311, 'train': 0.1976, 'validation': 0.2173}
Counts shrink after filtering; the old 20% share cap need 

## 5. Freeze the V3 dataset, exclusion audit and lineage

Save the detected-only subset, retained crop plan, excluded-event table, category mapping and V3 selection/preprocessing protocols. Copy the exact original camera/event preparation records and a reference snapshot of V2's selected manifests and detector decisions. Their hashes allow downstream notebooks to verify that V3 is exactly the intended crop-only subset.

The saved selection protocol records actual retained partition sizes rather than pretending that the original 1,500-event development budgets remain intact. Exclusion counts by partition and species, retained class counts, and post-exclusion camera contributions support reporting of the data flow and remaining imbalance.

The design record explicitly states that V3 followed earlier test inspection and retains V2 camera assignments. No fresh-camera generalization guarantee is implied. Published MegaDetector training may have included Caltech, so classifier camera holdout does not establish detector pretraining holdout. This notebook freezes a reproducible revision; image integrity and model performance are addressed separately in the following notebooks.


In [5]:
# Preserve the exact V2 preparation files: their protocol labels stay V2 because they ARE inherited.
upstream_names = ['image_manifest.csv','camera_split.json','excluded_images.csv',
                  'excluded_sequences.csv','event_manifest_all_categories.csv','preparation_run.json']
input_hashes = {}
for name in upstream_names:
    src, dst = PARENT_PREPARED / name, PREPARED / name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and sha(dst) != sha(src):
        raise ValueError(f'Existing inherited source differs: {dst}')
    if not dst.exists():
        shutil.copyfile(src, dst)
    input_hashes[name] = sha(dst)
snapshot = PREPARED / 'v2_reference'
snapshot.mkdir(exist_ok=True)
for name in ['protocol.json','selection_protocol.json','subset_manifest.csv',
             'training_ready_manifest.csv','crop_plan.csv','classes.json']:
    src, dst = PARENT_EXP / name, snapshot / name
    if dst.exists() and sha(dst) != sha(src):
        raise ValueError(f'Existing V2 reference differs: {dst}')
    if not dst.exists():
        shutil.copyfile(src, dst)
    input_hashes[f'v2_reference/{name}'] = sha(dst)

freeze(EXP / 'subset_manifest.csv', subset.to_csv(index=False))
freeze(EXP / 'crop_plan.csv', plan.to_csv(index=False))
freeze(EXP / 'excluded_fallback_events.csv', excluded.to_csv(index=False))
freeze(EXP / 'classes.json', jtext({'classes': CLASSES, 'class_to_idx': {c:i for i,c in enumerate(CLASSES)}}))
selection = dict(parent_selection)
selection.update(experiment_id=EXPERIMENT_ID, selection_version=3,
    parent_experiment_id=PARENT_EXPERIMENT_ID,
    sampling_rule='exact V2 selected events restricted to animal_crop; no replacement or resampling',
    evaluation_budgets={s:int(sizes[s]) for s in SPLITS if s != 'train'},
    parent_evaluation_budgets=parent_selection['evaluation_budgets'],
    retained_split_sizes={s:int(sizes[s]) for s in SPLITS},
    camera_cap_scope='inherited V2 sample before exclusion; actual post-exclusion shares are descriptive',
    input_sha256=input_hashes)
freeze(EXP / 'selection_protocol.json', jtext(selection))
protocol = copy_parent = json.loads(json.dumps(parent_protocol))
protocol.update(pipeline_kind='megadetector_detected_only_single_v3', protocol_version=3,
    experiment_id=EXPERIMENT_ID,
    parent_experiment_id=PARENT_EXPERIMENT_ID,
    scope='five known categories; one selected frame per event; conditional on qualifying MD animal detection',
    prior_test_exposure='V3 designed after V1/V2 test inspection; same camera assignments; revised sensitivity experiment, not independent confirmation')
protocol['crop_rule'].update(no_detection='exclude_event_no_replacement',
    plan_sha256=sha(EXP / 'crop_plan.csv'))
protocol['cache_preprocessing']['orientation'] = 'crop_requires_identity_EXIF'
protocol['input_sha256'] = {**input_hashes,
    **{f'{EXPERIMENT_ID}/{name}': sha(EXP / name) for name in
       ['selection_protocol.json','crop_plan.csv','subset_manifest.csv','classes.json','excluded_fallback_events.csv']}}
freeze(EXP / 'protocol.json', jtext(protocol))
freeze(EXP / 'local_source.json', jtext({'source_project_root': str(SOURCE_ROOT)}))
freeze(REPORT / 'exclusion_counts_by_split.csv', flow.to_csv())
freeze(REPORT / 'exclusion_counts_by_class_and_split.csv', by_class.to_csv(index=False))
freeze(REPORT / 'retained_class_counts.csv', counts.to_csv())
freeze(REPORT / 'camera_contribution_after_exclusion.csv', camera_counts.to_csv(index=False))
freeze(REPORT / 'V3_DESIGN.md',
    '# V3: detected-only revision\n\n'
    'Exact V2 selected-event subset with all full-frame fallbacks excluded from all five partitions. '
    'No new frame selection, replacement events or camera reassignment. Camera/preparation provenance '
    'files remain unchanged V2 records. V3 protocol and selection records identify the new experiment. '
    'The initial 20% development camera-share cap is inherited before exclusions; post-exclusion shares '
    'are recorded, not claimed to remain capped. All models start from the same ImageNet recipes, '
    'with new V3 checkpoints, temperatures and policies. V2 remains unchanged.\n\n'
    'Detection success does not establish semantic crop correctness. Excluded frames are no-detection '
    'events, not automatically verified empty scenes. Detection recall and unfiltered-stream coverage '
    'are outside scope. V3 coverage is conditional on retained detected events. Prior V1/V2 test '
    'exposure means this is a revised experiment, not independent confirmation on new cameras.\n')
print('Frozen V3 metadata:', EXP)
print('Next: run Notebook 02 V3, then start Notebook 03 V3.')


Frozen V3 metadata: /home/fia/Schreibtisch/wildlife_confidence_project_v3/data/prepared/five_categories_detected_only_v3
Next: run Notebook 02 V3, then start Notebook 03 V3.


## Continue to image verification

Run Notebook 02 V3 to copy and verify the retained PNGs from the complete local V2 cache. There is no need to repeat source-image downloading, MegaDetector inference or V2 model training. All six V3 classifiers will subsequently start from their specified ImageNet pretrained weights, with newly fitted calibration temperatures and acceptance policies.
